# PySpark inner join

## Source data

The source comes from jupyter-pyspark/f1-sourcefiles

# Inner the data from the results.csv and races.csv 


# Initalise a spark session

In [6]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


# Load results.csv and races.csv into dataframes

In [7]:
# Load csv files into dataframes
#  Contains headers
# Infers schema

results = spark.read.csv("f1-sourcefiles/results.csv", header=True, inferSchema=True)

races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True)


# Show the data type of the dataframes


In [8]:
# results 

results.printSchema()

# races

races.printSchema()

root
 |-- resultId: integer (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- number: string (nullable = true)
 |-- grid: integer (nullable = true)
 |-- position: string (nullable = true)
 |-- positionText: string (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: double (nullable = true)
 |-- laps: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: string (nullable = true)
 |-- fastestLap: string (nullable = true)
 |-- rank: string (nullable = true)
 |-- fastestLapTime: string (nullable = true)
 |-- fastestLapSpeed: string (nullable = true)
 |-- statusId: integer (nullable = true)

root
 |-- raceId: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- round: integer (nullable = true)
 |-- circuitId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- date: date (nullable = true)
 |-- time: string (nulla

# Use Alias in an INNER JOIN

In [17]:
# Inner join on same column name i.e. raceid


# Create and alias for a Dataframe just like tabel alias in SQL

results = results.alias("rslt")

races = races.alias("rcs")

# join on raceid and look at shuffle size to optimise on?


joined_df = results.join(races, col("rslt.raceId") == col("rcs.raceId"), how="inner") \
    .filter("rcs.year == 2024") \
    .select(col("rslt.resultId"), col("rcs.raceId"), col("rcs.year"), col("rcs.round"), col("rcs.name").alias("race_name"))

joined_df.show()

+--------+------+----+-----+------------------+
|resultId|raceId|year|round|         race_name|
+--------+------+----+-----+------------------+
|   26286|  1121|2024|    1|Bahrain Grand Prix|
|   26287|  1121|2024|    1|Bahrain Grand Prix|
|   26288|  1121|2024|    1|Bahrain Grand Prix|
|   26289|  1121|2024|    1|Bahrain Grand Prix|
|   26290|  1121|2024|    1|Bahrain Grand Prix|
|   26291|  1121|2024|    1|Bahrain Grand Prix|
|   26292|  1121|2024|    1|Bahrain Grand Prix|
|   26293|  1121|2024|    1|Bahrain Grand Prix|
|   26294|  1121|2024|    1|Bahrain Grand Prix|
|   26295|  1121|2024|    1|Bahrain Grand Prix|
|   26296|  1121|2024|    1|Bahrain Grand Prix|
|   26297|  1121|2024|    1|Bahrain Grand Prix|
|   26298|  1121|2024|    1|Bahrain Grand Prix|
|   26299|  1121|2024|    1|Bahrain Grand Prix|
|   26300|  1121|2024|    1|Bahrain Grand Prix|
|   26301|  1121|2024|    1|Bahrain Grand Prix|
|   26302|  1121|2024|    1|Bahrain Grand Prix|
|   26303|  1121|2024|    1|Bahrain Gran